# Space Debris Analysis Project


##### Phase 1: Problem Definition & Data Ingestion

* **Goal:** Define the objective and load raw data from its source without changing it.
* **Key Actions:**
  * Download data from APIs or files using `requests`
  * Load data into a DataFrame and review it with `df.head()` and `df.info()`



##### Phase 2: Data Cleaning & Preprocessing

* **Goal:** Fix errors and inconsistencies to make the dataset usable.
* **Key Actions:**
  * **Column Renaming:** Standardize column names by removing spaces and special characters
  * **Data Type Casting:** Convert columns to correct types (e.g., dates from `object` to `datetime64`)
  * **Duplicate Removal:** Find and remove duplicate rows
  * **Whitespace Cleaning:** Remove leading and trailing spaces from text
  * **Missing Values:** Decide to drop, fill, or keep missing values as appropriate



##### Phase 3: Exploratory Data Analysis (EDA)

* **Goal:** Find patterns, distributions, and relationships in the data using statistics and visualizations.
* **Key Actions:**
  * **Univariate Analysis:** Examine distribution of individual columns (e.g., histograms, value counts)
  * **Bivariate Analysis:** Compare relationships between variables (e.g., altitude vs. object type)
  * **Correlation Matrices:** Check how numerical variables relate to each other



##### Phase 4: Feature Engineering

* **Goal:** Create new variables from existing columns to improve analysis or modeling.
* **Key Actions:**
  * **Date Features:** Extract year, month, or calculate duration between dates
  * **Binning:** Group continuous variables into categories (e.g., Low vs. High orbit)
  * **Combined Features:** Create new metrics by combining existing ones



##### Phase 5: Modeling & Advanced Analysis (Optional)

* **Goal:** Apply machine learning or statistical models for predictions and insights.
* **Key Actions:** Split data, scale features, train models, and evaluate performance



##### Phase 6: Reporting & Deployment

* **Goal:** Share insights with stakeholders or create dashboards.
* **Key Actions:** Export clean data, build dashboards, or present findings




## Phase 1: Data Extracting and Saving the File

In [0]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import requests

### Setting up Options

In [0]:
%skip
# Check your full dataset
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)

display(df)


#### Note: HTTP Response Codes & Methods

**Response Codes Reference**

* **1xx**: Informational (request received, continuing process)
* **200**: Success
* **3xx**: Redirection (further action needed)
* **401**: Unauthorized
* **403**: Forbidden
* **404**: Not Found

**Diagnostic Code Checks**

* `r.status_code`
* `r.request.headers`
* `r.request.body`
* `r.headers`

**When to Use Different Response Methods**

* **`.json()`** - Use when API returns JSON data
  * Most modern REST APIs (GitHub, weather, etc.)
  * Common for web services and public APIs

* **`.text`** - Use for plain text responses
  * HTML pages
  * Raw string data

* **`.content`** - Use for binary data
  * Images
  * File downloads
  * Returns raw bytes that you can save to a file

In [0]:
%skip
# data extracted url
url = "https://celestrak.org/pub/satcat.csv"

# path where you want to save the file
path = "/Workspace/Users/guruvendra47@gmail.com/Space-Debris-Project/data/raw"
# path = "/Workspace/Users/guruvendra47@gmail.com/space-debris-project/space_debris_raw.csv" when you want code simpler and dont want filname seperated variable
# file name which you want to give
filename = "space_debris_raw.csv"
# combine full path with your filename
full_path = f"{path}/{filename}"


try:
    # getting data from url
    response = requests.get(url)

    # check if it is success or failed
    if response.status_code == 200:
        with open(full_path, "wb") as f:
            f.write(response.content)
            print(f"File saved successfully to: {full_path}")
    else:
        print(f"Failed to download. Status code: {response.status_code}")
except requests.exceptions.RequestException as e:
    print(f"An error occurred: {e}")


## Phase 2: Data Loadig & Initial Exploration & Formatting

In [0]:
path = "/Workspace/Users/guruvendra47@gmail.com/Space-Debris-Project/data/raw/space_debris_raw.csv"
df = pd.read_csv(path)
df

### Basic Pandas Operations
- .head()
- .tail()
- .shape
- .columns
- .info()
- .describe()
- .rename(column=oldcolumnname, newcolumnname)

In [0]:
# head
df.head(10)

In [0]:
df.tail(10)

In [0]:
df.shape

In [0]:
# df.info()
df.dtypes

In [0]:
df.columns

#### Notes
**Column Headers**

* **`rename()`**: Changes header titles.

**Row Values**

* **`replace()`**: Use for quick code without a loop.
* **`map().fillna()`**: Use with a loop for max speed.

In [0]:
# rename the columns
rename_mapping={
        "OBJECT_NAME": "ObjectName",
        "OBJECT_ID": "ObjectID",
        "NORAD_CAT_ID": "CatalogID",
        "OBJECT_TYPE": "ObjectType",
        "OPS_STATUS_CODE": "OperationalStatus",
        "OWNER": "Owner",
        "LAUNCH_DATE": "LaunchDate",
        "LAUNCH_SITE": "LaunchSite",
        "DECAY_DATE": "DecayDate",
        "PERIOD": "OrbitalPeriodMin",
        "INCLINATION": "InclinationDegrees",
        "APOGEE": "MaxAltitudeKM",
        "PERIGEE": "MinAltitudeKM",
        "RCS": "RadarSizeSQM",
        "DATA_STATUS_CODE": "DataStatus",
        "ORBIT_CENTER": "OrbitCenter",
        "ORBIT_TYPE": "OrbitState",
    }

df = df.rename(columns=rename_mapping)

print(df.columns.tolist())

In [0]:
df.head(4)

### Understanding the Data
#### Space Debris Dataset Column Descriptions

* **`satellite_name`**: This columns contain names of satellite, rocket stage, or space debris.
* **`satellite_id`**: This is Unique No assigned to satellite, rocket stage, or space debris in International Designator (COSPAR ID) format as `YYYY-NNNAA` (Launch Year, Launch Number, Piece of launch).
* **`norad_cat_id`**: Unique tracking catalog number assigned by USSPACECOM/NORAD.
* **`satellite_type`**: Classification of the object:
  * `PAY`: Payload (active or inactive satellite).
  * `R/B`: Rocket Body (spent upper stage left in orbit).
  * `DEB`: Debris (fragmentation or dropped hardware).
* **`operation_status_code`**: Operational status indicator:
  * `+`: Operational/Active.
  * `-`: Non-operational/Inactive.
  * `P` : Partially Operational / Standby
  * `D`: Decayed (re-entered Earth's atmosphere).
  * `NaN`: Unknown or unassigned status.
* **`country`**: Country, space agency, or commercial owner.
* **`launch_date`**: Date the object was launched into space.
* **`launch_location`**: launch site/spaceport.
* **`decay_date`**: Date in which the object re-entered Earth's atmosphere. Missing values (`NaN`/`NaT`) mean the object is still in orbit.
* **`period_time_min`**: Orbital period in minutes (time required to complete one full orbit around Earth).
* **`Inclination`**: Angle (in degrees) between the orbital plane and Earth's equator.
* **`highest_altitude_km`**: Highest point of the orbit from Earth's surface (Apogee in km).
* **`lowest_altitude_km`**: Lowest point of the orbit from Earth's surface (Perigee in km).
* **`radar_cross_section`**: Radar Cross Section (\text{m}^2), representing the reflective physical size detected by radar.
* **`data_status_code`**: Administrative code indicating tracking data reliability.
* **`orbit_center`**: Body being orbited (`EA` = Earth) etc.
* **`orbit_type`**: Orbital state (`ORB` = Currently orbiting, `IMP` = Impacted/Decayed).



## Phase 3. Data Cleaning and Wrangling

### Step 1: Finding and Removing Duplicates
1. Identify duplicate rows  in the dataset.
2. Use suitable techniques to remove duplicate rows and verify the removal.
3. Summarize how to handle missing values appropriately.
4. Use ConvertedCompYearly to normalize compensation data.

In [0]:
# df.duplicated().value_counts()
df.duplicated().sum()

# to see duplicate
df[df.duplicated()]

# df.duplicated(subset=columnslist, keep=False) in order to check how many same values are there over the data 

# Removing the Duplicates
#df.drop_duplicates()

#### Note: No Duplicate Values Found

### Step 2: Removing Whitespaces
##### Note: 
- We only check string columns (dtype == 'object') because numeric columns (int, float) cannot contain leading/trailing whitespaces.

In [0]:
# step 1: Remove whitespaces

# Before: Count rows with whitespaces in each column
print("=== Before: Whitespace Analysis ===")
for i in df.columns:
    if df[i].dtype == 'object':
        wht_spc = (df[i].notna()) & (df[i] != df[i].str.strip())
        cnt_wht = wht_spc.sum()
        if cnt_wht > 0:
            print(f"{i}: {cnt_wht}")
        else:
            print(f"{i}: {cnt_wht}")

print("\nRemoving....")
for i in df.columns:
    if df[i].dtype == 'object':
        df[i] = df[i].str.strip()

# After: Count rows with whitespaces in each column
print("\n=== After: Whitespace Analysis ===")
for i in df.columns:
    if df[i].dtype == 'object':
        wht_spc = (df[i].notna()) & (df[i] != df[i].str.strip())
        cnt_wht = wht_spc.sum()
        if cnt_wht > 0:
            print(f"{i}: {cnt_wht} ")
        else:
            print(f"{i}: {cnt_wht}")


### Step 3: Changing Data Types
- check which columns data type is wrong 
- change into "object", "int64", float64", "datatime64[ns]"
- we found that LaunchDate and DecayDate both columns dtypes need to change from object to Datetime

In [0]:
# check which columns has wrong data type
print("=== Before ===")
display(df.info())

print("\nChanging....")
df["LaunchDate"] = pd.to_datetime(df["LaunchDate"],format="mixed", errors= "coerce")
df["DecayDate"] = pd.to_datetime(df["DecayDate"],format="mixed", errors="coerce")

print("\n\n=== After ===")
print(df.dtypes)


### Step 4: Finding and Handling Missing Values
1. Identifying the missing values and converting in percentage for better view
2. Visualize missing values using a heatmap
3. Count the number of missing rows for a specific column (.value_counts())
4. Identifing the most frequent or more time repeated value in a spcific column
5. Taking decision to drop the rows or column or fill with mean, median, .idxmax()(most repeated value), .ffill()(forward-fill is filling with next value)
6. Visualize the distribution of a column after imputationF

In [0]:
# identifing the missing values
msgvalue = df.isnull().sum()

# In percentage for better view and rounded to 2 decimal 
prt = (msgvalue / len(df)*100).round(2)

# putting side by side for better view
dic = pd.DataFrame({"Missingvalues": msgvalue, "Percentage (%)":prt})
print(dic)

#### Note: Missing Values Summary
- OperationalStatus
- DecayDate
- OrbitalPeriodMinutes
- InclinationDegrees
- ApogeeKM
- PerigeeKM 
- RadarCrossSectionSQM
- DataStatusCode


#### Column: OperationalStatus
#### Column: OperationalStatus

Missing values occur because space tracking agencies only assign status codes to active/inactive satellites. used upper rocket bodies and debris do not have operational missions, so their status is naturally left blank so we filling that as `"Unknown"`.

In [0]:
# fillna NaN with Unknown for operation status code
df["OperationalStatus"] = df["OperationalStatus"].fillna("Unknown")
print(df.isnull().sum())

#### Column: DecayDate
#### Column: DecayDate
 - A missing `DecayDate` (`NaT`) means the object has not re-entered Earth's atmosphere and is still currently in orbit. We keep these values empty to maintain the `datetime64` data type so date calculations do not break. In Power BI, null values will be used to filter and render all active objects around Earth.

In [0]:

# DecayDate kept as NaT - missing values mean the object is still in space. Since it's not a string, I left NaT rather than changing it to "in space orbit".


#### Columns: Orbital Parameters

**Columns:** OrbitalPeriodMin, InclinationDegrees, MaxAltitudeKM, MinAltitudeKM

- Orbital parameters cannot be filled with zero (0 minutes = invalid orbit).
- In this case, we have to fill NaN values with median values, grouped by `ObjectType` and filtered by `OrbitCenter`.
- This ensures Earth-orbiting payloads use Earth payload medians, etc.

In [0]:
df["OrbitCenter"].unique()

In [0]:

# columns : period_time_min, Inclination, highest_altitude_km, lowest_altitude_km

cols = ["OrbitalPeriodMin", "InclinationDegrees", "MaxAltitudeKM", "MinAltitudeKM"]


orb = df["OrbitCenter"].unique()
# filtering data
for i in orb:
    mask = df["OrbitCenter"] == i
    # grouping the data
    for j in cols:
        df.loc[mask, j] = df.loc[mask, j].fillna(df[mask].groupby("ObjectType")[j].transform("median"))

print("\n=== Remaining Missing Values ===")
print(df.isnull().sum())

#### Note: Remaining Nulls values After Group Median
- After filtering with OrbitCenter and groupby ObjectType, 67 nulls remain in the "OrbitalPeriodMinutes", "OrbitalTiltDegrees", "HighestPointKm", and "LowestPointKm" columns because the mean could not be calculated because these columns value is null.
- As orbital data, making group medians NaN. We fill them with -1 (sentinel value) to keep numeric dtypes, preserve row count.

In [0]:
# 67 nan we will with sentinel value -1 as it non-earth orbit

df[cols] = df[cols].fillna(-1)
print(df.isnull().sum())


#### Column: RadarCrossSectionSQM
#### Note: RadarSizeSQM Null Handling

- Radar Cross Section (RCS) is missing for objects that are too small or far away to be measured by ground radar. We fill missing values with `0.0` to treat them as untracked physical sizes without breaking numeric models.

In [0]:
# radar_cross_section column
df["RadarSizeSQM"] = df["RadarSizeSQM"].fillna(0.0)
print(df.isnull().sum())


#### Column: DataStatus 

- According to space tracking documentation (CelesTrak SATCAT Documentation), 
- the official meanings of DataStatusCodes are:
   - NIE: No Initial Elements (Sensors detected the object at launch, but stable initial orbital calculations could not be established)
   - NEA: No Elements Available (Tracking elements are missing or discontinued)
   - NCE: No Current Elements (Historical tracking exists, but current active updates are unassigned)
   - NaN (Blank): Nominal Tracking or Active Elements Available (The standard baseline state for cataloged objects)

- Strategy: Replace codes with descriptive labels and fill NaN with "Active Tracking"

In [0]:
# 98 percentage is active tracking data which is null 

stcd = {"NIE":"No Initial Elements", "NEA":"No Elements Available", "NCE": "No Current Elements"}
df["DataStatus"] = df["DataStatus"].replace(stcd)
df["DataStatus"] = df["DataStatus"].fillna("Active Tracking")
print(df.isnull().sum())
df

## Step 5: Data Standardization

- Standardization is the process of transforming data into a common format, allowing the researcher to make the meaningful comparison

#### Columns to Standardize:

- ObjectType 
- OperationalStatus
- Owner 
- LaunchSite
- OrbitCenter
- OrbitState
- DataStatus.



### Column Modification Methods

**Column Headers**

* **`rename()`**: Changes header titles.

**Row Values**

* **`replace()`**: Use for quick code without a loop.
* **`map().fillna()`**: Use with a loop for max speed.

### .map() vs .replace()

**Use `.map()`:**
- When remapping an entire column into a new scale
- Unlisted values automatically turn to NaN
- Best for complete transformations

**Use `.replace()`:**
- When fixing a couple of specific values
- Everything else stays as-is
- Best for targeted replacements

### Columns: ObjectType & OperationalStatus

In [0]:

# No 1
# Column: ObjectType
dic = {"R/B":"Rocket Body", "PAY":"Payload", "DEB": "Debris"}
df["ObjectType"] = df["ObjectType"].replace(dic)
df.head()

# No2
# Column: OperationalStatus
dic = {"D": "Decayed (D)", "+": "Operational (+)", "-":"Non-Operational (-)", "P": "Partially Operational (P)"  }
df["OperationalStatus"] = df["OperationalStatus"].replace(dic)
df



### Column: Owner

In [0]:
# check Unique Country code
df["Owner"].unique()

In [0]:
owners = {
    "AB": "ARABSAT (Arab Satellite)",
    "AC": "Asia-Pacific Space Cooperation",
    "ABS": "ABS (Asia Broadcast Satellite)",
    "ALG": "Algeria",
    "ANG": "Angola",
    "ARGN": "Argentina",
    "ARM": "Armenia",
    "ASRA": "AsiaSat Telecommunications",
    "AUS": "Australia",
    "AZER": "Azerbaijan",
    "BEL": "Belgium",
    "BELA": "Belarus",
    "BGD": "Bangladesh",
    "BHR": "Bahrain",
    "BHUT": "Bhutan",
    "BOL": "Bolivia",
    "BRAZ": "Brazil",
    "BUL": "Bulgaria",
    "BWA": "Botswana",
    "CA": "Canada",
    "CHBZ": "China / Brazil (CBERS)",
    "CHLE": "Chile",
    "CIS": "Russia / Former USSR",
    "COL": "Colombia",
    "CRI": "Costa Rica",
    "CZCH": "Czech Republic",
    "DEN": "Denmark",
    "DJI": "Djibouti",
    "ECU": "Ecuador",
    "EGYP": "Egypt",
    "ESA": "European Space Agency",
    "ESRO": "European Space Research Org",
    "EST": "Estonia",
    "ETH": "Ethiopia",
    "EUME": "EUMETSAT",
    "EUTE": "EUTELSAT",
    "FGER": "West Germany (Former)",
    "FIN": "Finland",
    "FR": "France",
    "FRIT": "France / Italy",
    "GER": "Germany",
    "GHA": "Ghana",
    "GLOB": "Globalstar (Commercial)",
    "GREC": "Greece",
    "GRSA": "South Africa",
    "GUAT": "Guatemala",
    "HRV": "Croatia",
    "HUN": "Hungary",
    "IM": "Inmarsat (Commercial)",
    "IND": "India",
    "INDO": "Indonesia",
    "IRAN": "Iran",
    "IRAQ": "Iraq",
    "IRL": "Ireland",
    "ISRA": "Israel",
    "ISS": "International Space Station",
    "IT": "Italy",
    "ITSO": "INTELSAT (Commercial)",
    "JOR": "Jordan",
    "JPN": "Japan",
    "KAZ": "Kazakhstan",
    "KEN": "Kenya",
    "KWT": "Kuwait",
    "LAOS": "Laos",
    "LKA": "Sri Lanka",
    "LTU": "Lithuania",
    "LUXE": "Luxembourg",
    "MA": "Morocco",
    "MALA": "Malaysia",
    "MCO": "Monaco",
    "MDA": "Moldova",
    "MEX": "Mexico",
    "MMR": "Myanmar (Burma)",
    "MNE": "Montenegro",
    "MNG": "Mongolia",
    "MUS": "Mauritius",
    "NATO": "NATO",
    "NETH": "Netherlands",
    "NICO": "New ICO",
    "NIG": "Nigeria",
    "NKOR": "North Korea",
    "NOR": "Norway",
    "NPL": "Nepal",
    "NZ": "New Zealand",
    "O3B": "O3b Networks / SES",
    "ORB": "Orbcomm (Commercial)",
    "PAKI": "Pakistan",
    "PERU": "Peru",
    "POL": "Poland",
    "POR": "Portugal",
    "PRC": "China",
    "PRY": "Paraguay",
    "QAT": "Qatar",
    "RASC": "RASCOMSTAR-QAF",
    "ROC": "Taiwan",
    "ROM": "Romania",
    "RP": "Philippines",
    "RWA": "Rwanda",
    "SAFR": "South Africa",
    "SAUD": "Saudi Arabia",
    "SDN": "Sudan",
    "SEAL": "Sea Launch",
    "SEN": "Senegal",
    "SES": "SES S.A. (Commercial)",
    "SGJP": "Singapore / Japan",
    "SING": "Singapore",
    "SKOR": "South Korea",
    "SLB": "Solomon Islands",
    "SPN": "Spain",
    "STCT": "Singapore / Taiwan (ST-1)",
    "SVK": "Slovakia",
    "SVN": "Slovenia",
    "SWED": "Sweden",
    "SWTZ": "Switzerland",
    "TBD": "Unassigned / To Be Determined",
    "THAI": "Thailand",
    "TMMC": "Turkmenistan / Monaco",
    "TUN": "Tunisia",
    "TURK": "Turkey",
    "UAE": "United Arab Emirates",
    "UGA": "Uganda",
    "UK": "United Kingdom",
    "UKR": "Ukraine",
    "URY": "Uruguay",
    "US": "United States",
    "USBZ": "United States / Brazil",
    "VAT": "Vatican City",
    "VENZ": "Venezuela",
    "VTNM": "Vietnam",
    "ZWE": "Zimbabwe",
}

df["Owner"] = df["Owner"].replace(owners)

df

#### making copy of that file

In [0]:
df = df.copy()

### Column: LaunchSite 


In [0]:
df["LaunchSite"].unique()

In [0]:
sitemapping = {
    "AFETR": "United States (AFETR)",
    "AFWTR": "United States (AFWTR)",
    "CAS": "Spain (CAS)",
    "DLS": "Russia (DLS)",
    "ERAS": "United States (ERAS)",
    "FRGUI": "French Guiana (FRGUI)",
    "HGSTR": "Algeria (HGSTR)",
    "JJSLA": "South Korea (JJSLA)",
    "JSC": "China (JSC)",
    "KODAK": "United States (KODAK)",
    "KSCUT": "Japan (KSCUT)",
    "KWAJ": "Marshall Islands (KWAJ)",
    "KYMSC": "Russia (KYMSC)",
    "NSC": "South Korea (NSC)",
    "PLMSC": "Russia (PLMSC)",
    "RLLB": "New Zealand (RLLB)",
    "SCSLA": "China (SCSLA)",
    "SEAL": "International (SEAL)",
    "SEMLS": "Iran (SEMLS)",
    "SMTS": "Iran (SMTS)",
    "SNMLP": "Kenya (SNMLP)",
    "SRILR": "India (SRILR)",
    "SUBL": "International (SUBL)",
    "SVOBO": "Russia (SVOBO)",
    "TAISC": "China (TAISC)",
    "TANSC": "Japan (TANSC)",
    "TYMSC": "Kazakhstan (TYMSC)",
    "VOSTO": "Russia (VOSTO)",
    "WLPIS": "United States (WLPIS)",
    "WOMRA": "Australia (WOMRA)",
    "WRAS": "United States (WRAS)",
    "WSC": "China (WSC)",
    "XICLF": "China (XICLF)",
    "YAVNE": "Israel (YAVNE)",
    "YSLA": "China (YSLA)",
    "YUN": "North Korea (YUN)",
}

df["LaunchSite"] = df["LaunchSite"].map(sitemapping).fillna(df["LaunchSite"])
# or
df["LaunchSite"] = df["LaunchSite"].replace(sitemapping)
df.head(20)

### Column: OrbitCenter

In [0]:
df["OrbitCenter"].unique()

In [0]:
orbname = {
    # Planets & Celestial Bodies
    "ME": "Mercury (ME)",
    "VE": "Venus (VE)",
    "EA": "Earth (EA)",
    "MA": "Mars (MA)",
    "JU": "Jupiter (JU)",
    "SA": "Saturn (SA)",
    "UR": "Uranus (UR)",
    "NE": "Neptune (NE)",
    "PL": "Pluto (PL)",
    "SU": "Sun (SU)",
    "MO": "Moon (MO)",
    "CO": "Comet (CO)",
    # Systems & Lagrange Points
    "EM": "Earth-Moon System (EM)",
    "SS": "Solar System (SS)",
    "AS": "Asteroids (AS)",
    "EL": "Earth-Moon Lagrange (EL)",
    "EL1": "Earth-Moon L1 (EL1)",
    "EL2": "Earth-Moon L2 (EL2)",
    # Specific High-Profile Space Stations
    "25544": "ISS (25544)",
    "28358": "Tiangong Target (28358)",
    "48274": "Tianhe Core (48274)",
}

df["OrbitCenter"] = df["OrbitCenter"].replace(orbname)
df

### Column: OrbitState

In [0]:
df["OrbitState"].unique()

In [0]:
obste = {
    # Existing Array Codes
    "IMP": "Impact (IMP)",
    "ORB": "Orbiter (ORB)",
    "LAN": "Lander (LAN)",
    "DOC": "Dock / Rendezvous (DOC)",
    # Other CelesTrak Mission Types
    "FLY": "Flyby (FLY)",
    "SAMP": "Sample Return (SAMP)",
    "CREW": "Crewed (CREW)",
    "CARG": "Cargo / Resupply (CARG)",
    "DEP": "Deployer (DEP)",
    "REL": "Relay (REL)",
}

df["OrbitState"] = df["OrbitState"].replace(obste)
df

# Phase 4: Exploratory Data Analysis (EDA)
- In this phase, we analyze distributions, feature relationships, correlations, and historical launch trends across the cleaned space debris dataset.

**1. Univariate Analysis (One Variable)**

*Looking at columns individually to understand distributions, spot outliers, and check skewness.*

* **Histograms & Boxplots** — Use histograms (with KDE) to see the spread of numerical data; use boxplots to spot extreme outliers.
* **The `describe()` Check** — Run `df.describe().T` on numerical columns. It's a quick sanity check on mean, min, max, and quartiles.
* **Categorical Breakdown** — Run `.value_counts()` or plot bar charts / pie charts on text columns to see the volume of each category.

**2. Bivariate Analysis (Two Variables)**

*Exploring relationships between pairs of variables.*

* **Scatter Plots** — Best for seeing if one variable goes up when another goes down (e.g., perigee vs. apogee).
* **Box Plots by Category** — Compare a numerical variable across groups (e.g., orbital period by object type).
* **Correlation & Pivot Tables** — Run pairwise correlation to see which numerical columns move together. Use pivot tables to group data and find averages across two categories.

**3. Multivariate Analysis (Three+ Variables)**

*Adding a grouping or temporal dimension to uncover deeper patterns.*

* **Color-Coded Scatter Plots** — Take a scatter plot and color-code by a category (hue) to see deeper patterns.
* **Correlation Heatmaps** — Visualize multiple numerical variables at once to detect multicollinearity.
* **Time Series** — Plot metrics over time (e.g., launches per year by object type) to find trends, seasonal spikes, or accumulation patterns.
* **Domain-Specific Questions** — Answer the actual research questions (e.g., the ratio of debris to active satellites, top countries by launch volume).

## 4.1 Univariate Analysis



#### Numeric Variables
- Summary stats: count, mean, median, std, min, max, quartiles, IQR, skewness
- Visuals: histogram, boxplot, density plot
- Check for multimodality (multiple peaks) and heavy tails (outliers)

#### Categorical Variables
- Frequency counts
- Bar charts and Pareto charts
- Check rare categories (low-frequency levels) and cardinality

#### Date/Time Variables
- Check for time range, gaps, time granularity
- Visuals: time series plot, seasonality checks, heatmap (for example: day vs hour), bar chart (month/year)

#### Text Variables
- Token counts, most common words, length distributions
- Histogram of text lengths
- Word cloud / bar of top words

#### 4.1.a Numberic Variable

In [0]:
# Number Variable

num = df.select_dtypes(include=["number"])

summary = num.describe().T
summary["median"] = num.median()
# summary = summary.rename(columns={"50%": "median"})
summary["IQR"] = summary["75%"] - summary["25%"]
summary["skewness"] = num.skew()

print("Numeric Summary Statistics:")
summary.round(2)


## Summary Findings from describe()

1. **Row Count**: The dataset contains 70,586 rows across all numerical columns, with no missing values after imputation.

2. **Outliers Present**: Standard deviation values are very high relative to the mean in several columns (especially `MaxAltitudeKM` with std = 167,604 km and `OrbitalPeriodMin` with std = 19,840 min), indicating the presence of extreme outliers.

3. **High Variability**: Standard deviation measures how far data points scatter from the mean. Large std values indicate wide spread and heterogeneous data distribution.

4. **Placeholder Values**: Minimum values of -1.0 in orbital columns (`OrbitalPeriodMin`, `InclinationDegrees`, `MaxAltitudeKM`, `MinAltitudeKM`) indicate missing data placeholders that need to be addressed.

5. **Right-Skewed Distributions**: Most orbital parameters show heavy positive skewness, meaning data is clustered near lower values with long tails extending to extreme highs (typical of space objects concentrated in LEO with few deep-space outliers).

6. **IQR Analysis**: The Interquartile Range (Q3 - Q1) shows that the middle 50% of satellites operate in relatively narrow bands, while outliers drive the high maximums and standard deviations. 

In [0]:
# Visualization - all numeric columns (sampled for speed)

num_cols = list(df.select_dtypes(include=["number"]).columns)

# Sample 10K rows for plotting — visually identical to full 70K
plot_df = df.sample(min(len(df), 10000), random_state=42)

fig, axes = plt.subplots(len(num_cols), 2, figsize=(12, 4 * len(num_cols)), squeeze=False)

for idx, col in enumerate(num_cols):
    data = plot_df[col].dropna()

    # Histogram + KDE
    sns.histplot(data, bins=50, kde=True, color='teal', ax=axes[idx, 0])
    axes[idx, 0].set_title(f'Distribution: {col}')
    axes[idx, 0].set_xlabel(col)

    # Boxplot
    sns.boxplot(y=data, color="blue", ax=axes[idx, 1])
    axes[idx, 1].set_title(f'Boxplot: {col}')
    axes[idx, 1].set_ylabel(col)

plt.tight_layout()
plt.show()

print(f"\nVisualized {len(num_cols)} numerical columns: {num_cols}")

#### Categorical Variables

In [0]:
# Analysing Categorical Variables

cat_cols = df.select_dtypes(include=['object']).columns.tolist()

for i in cat_cols:
    print(f"Column: {i}")
   
    frq = df[i].value_counts()
    prt = (df[i].value_counts(normalize=True) * 100).round(2)
    
    dtfr = pd.DataFrame({"Frequency Counts": frq, "Percentages": prt})
    print(dtfr)
    print("\n")

# Bar Charts and Pareto Charts for Categorical Variables
fig, axes = plt.subplots(len(cat_cols), 2, figsize=(16, 5 * len(cat_cols)), squeeze=False)

for idx, col in enumerate(cat_cols):
    # Get value counts
    value_counts = df[col].value_counts()
    
    # Bar Chart
    ax1 = axes[idx, 0]
    sns.barplot(x=value_counts.values, y=value_counts.index, ax=ax1, palette='viridis')
    ax1.set_title(f'Bar Chart: {col}', fontsize=12, fontweight='bold')
    ax1.set_xlabel('Count')
    ax1.set_ylabel(col)
    for i, v in enumerate(value_counts.values):
        ax1.text(v + max(value_counts.values) * 0.01, i, f'{v:,}', va='center')
    
    # Pareto Chart
    ax2 = axes[idx, 1]
    ax2_twin = ax2.twinx()
    
    # Sort by frequency (descending)
    sorted_counts = value_counts.sort_values(ascending=False)
    cumulative_pct = (sorted_counts.cumsum() / sorted_counts.sum() * 100)
    
    # Bar chart for counts
    bars = ax2.bar(range(len(sorted_counts)), sorted_counts.values, color='skyblue', alpha=0.7)
    ax2.set_xlabel('Categories (sorted by frequency)')
    ax2.set_ylabel('Count', color='skyblue')
    ax2.tick_params(axis='y', labelcolor='skyblue')
    ax2.set_xticks(range(len(sorted_counts)))
    ax2.set_xticklabels(sorted_counts.index, rotation=45, ha='right')
    
    # Line chart for cumulative percentage
    ax2_twin.plot(range(len(sorted_counts)), cumulative_pct.values, color='red', marker='o', linewidth=2, label='Cumulative %')
    ax2_twin.set_ylabel('Cumulative Percentage (%)', color='red')
    ax2_twin.tick_params(axis='y', labelcolor='red')
    ax2_twin.set_ylim([0, 105])
    ax2_twin.axhline(y=80, color='green', linestyle='--', linewidth=1, alpha=0.7, label='80% Line')
    ax2_twin.legend(loc='lower right')
    
    ax2.set_title(f'Pareto Chart: {col}', fontsize=12, fontweight='bold')
    ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nGenerated Bar Charts and Pareto Charts for {len(cat_cols)} categorical columns.")


### Outlier Detection - Box Plots

In [0]:
# Select only numeric columns
num_col = df.select_dtypes(include=["number"])

# Check quartiles for outlier detection
Q1 = num_col.quantile(0.25)
Q3 = num_col.quantile(0.75)
IQR = Q3 - Q1

# Define lower and upper bounds
lower_value = Q1 - 1.5 * IQR
upper_value = Q3 + 1.5 * IQR

# Identify outliers
outliers = (num_col < lower_value) | (num_col > upper_value)
outlier_counts = outliers.sum()

print("Outlier Counts by Column:")
print(outlier_counts)
print(f"\nTotal rows with outliers: {outliers.any(axis=1).sum()}")


# visulization

# Calculate number of columns and rows needed for subplots
num_cols = len(df.columns)
num_rows = (num_cols + 2) // 3  # 3 columns per row

plt.figure(figsize=(15, num_rows * 4))

for index, col in enumerate(df.columns, 1):
    plt.subplot(num_rows, 3, index)
    sns.boxplot(data=df, y=col)
    plt.title(f'Boxplot of {col}')
    plt.ylabel(col)

plt.tight_layout()
plt.show()


### Distribution Histograms with KDE

KDE (Kernel Density Estimation) can only be used with continuous numerical data (like measurements, weights, altitudes, or speeds)—never with categories, text labels, or discrete counts.

**Charts Where KDE Can Be Used**

* Histograms (`sns.histplot`)
* Probability Density Plots (`sns.kdeplot`)
* Distribution Plots (`sns.displot`)

In [0]:
sns.set_theme(style="whitegrid")
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Distribution of Inclination
sns.histplot(df['InclinationDegrees'].dropna(), bins=50, kde=True, ax=axes[0, 0], color='teal')
axes[0, 0].set_title('Distribution of Orbit Inclination (Degrees)')

# 2. Distribution of Min Altitude (Perigee) - Log Scale
sns.histplot(df['MinAltitudeKM'].dropna(), bins=50, log_scale=(False, True), ax=axes[0, 1], color='coral')
axes[0, 1].set_title('Distribution of Min Altitude / Perigee (km)')

# 3. Distribution of Max Altitude (Apogee)
sns.histplot(df['MaxAltitudeKM'].dropna(), bins=50, kde=True, ax=axes[1, 0], color='skyblue')
axes[1, 0].set_title('Distribution of Max Altitude / Apogee (km)')

# 4. Distribution of Radar Cross Section (Physical Size)
sns.histplot(df['RadarSizeSQM'].dropna(), bins=50, log_scale=(False, True), ax=axes[1, 1], color='purple')
axes[1, 1].set_title('Radar Cross Section Size (SQM)')

plt.tight_layout()
plt.show()

## 4.2 .Describe() Statistical Summary
- count 
- mean
- std
- min
- max
- percentiles

In [0]:
# Select numerical columns
numeric_df = df.select_dtypes(include=['float64', 'int64'])

# Generate descriptive statistics
statistical_summary = numeric_df.describe().round(2)

print("=== Descriptive Statistical Summary ===")
print(f"\nTotal records: {len(df):,}")
print(f"Numerical columns: {len(numeric_df.columns)}\n")

print(statistical_summary)

# Additional statistics
print("\n=== Additional Statistics ===")
for i in numeric_df.columns:
    print(f"\n{i}:")
    print(f"  Skewness: {numeric_df[i].skew():.2f}")
    print(f"  Kurtosis: {numeric_df[i].kurtosis():.2f}")
    print(f"  Missing values: {numeric_df[i].isna().sum()}")

## 4.3 Bivariate Analysis

- Exploring relationships between pairs of variables to detect orbital clusters (LEO, MEO, GEO)
- compare metrics across object types

### Scatter Plot: Perigee vs Apogee

In [0]:
plt.figure(figsize=(10, 6))
sns.scatterplot( data=df, x='MinAltitudeKM', y='MaxAltitudeKM', hue='ObjectType', alpha=0.6, palette='Set1')
plt.xscale('log')
plt.yscale('log')
plt.title('Perigee (Min Altitude) vs. Apogee (Max Altitude)')
plt.xlabel('Perigee / Min Altitude (km)')
plt.ylabel('Apogee / Max Altitude (km)')
plt.legend(title='Object Type')
plt.grid(True, alpha=0.3)
plt.show()

### Box Plot: Orbital Period by Object Type

In [0]:
plt.figure(figsize=(12, 6))
sns.boxplot(data=df, x='ObjectType', y='OrbitalPeriodMin', palette='Set2')
plt.title('Orbital Period Distribution by Object Type', fontsize=14, fontweight='bold')
plt.xlabel('Object Type')
plt.ylabel('Orbital Period (Minutes)')
plt.yscale('log')
plt.grid(True, alpha=0.3, axis='y')
plt.show()

## 4.4 Correlation Analysis & Pivot Tables

*Checking for multicollinearity among numerical features and examining categorical relationships*

### Correlation Heatmap

In [0]:
plt.figure(figsize=(10, 8))
numeric_cols = ['OrbitalPeriodMin', 'InclinationDegrees', 'MaxAltitudeKM', 'MinAltitudeKM', 'RadarSizeSQM']
corr_matrix = df[numeric_cols].corr(method='spearman')

sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5, 
            square=True, cbar_kws={'shrink': 0.8})
plt.title('Spearman Rank Correlation Matrix of Orbital Parameters', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Print strong correlations
print("\n=== Strong Correlations (|r| > 0.7) ===")
for i in range(len(corr_matrix.columns)):
    for j in range(i+1, len(corr_matrix.columns)):
        if abs(corr_matrix.iloc[i, j]) > 0.7:
            print(f"{corr_matrix.columns[i]} <-> {corr_matrix.columns[j]}: {corr_matrix.iloc[i, j]:.3f}")

In [0]:
# Create pivot table for average orbital parameters
grouped_pivot = df.groupby(['ObjectType', 'OrbitState'])[['OrbitalPeriodMin', 'MaxAltitudeKM']].mean().reset_index()
pivot_table = grouped_pivot.pivot(index='ObjectType', columns='OrbitState', values='MaxAltitudeKM')

print("=== Pivot Table: Average Max Altitude (km) by Object Type & Orbit State ===")
print(pivot_table)

# Additional cross-tabulation
print("\n=== Cross-Tabulation: Object Count by Type & Orbit State ===")
crosstab = pd.crosstab(df['ObjectType'], df['OrbitState'], margins=True)
print(crosstab)

## 4.5 Historical Time-Series Trends

*Analyzing historical launch volume and object creation trends across decades*

In [0]:
# Extract Launch Year
df['LaunchYear'] = df['LaunchDate'].dt.year

# Create stacked area chart
plt.figure(figsize=(14, 7))
launch_trends = df.groupby(['LaunchYear', 'ObjectType']).size().unstack(fill_value=0)
launch_trends.plot(kind='area', stacked=True, alpha=0.8, figsize=(14, 7), colormap='tab10')

plt.title('Historical Objects Launched / Created Over Time by Type', fontsize=14, fontweight='bold')
plt.xlabel('Launch Year', fontsize=12)
plt.ylabel('Object Count', fontsize=12)
plt.legend(title='Object Type', loc='upper left')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Print key statistics
total_launches = df.groupby('LaunchYear').size()
print(f"\n=== Launch Statistics ===")
print(f"Total objects: {len(df):,}")
print(f"Peak year: {total_launches.idxmax()} with {total_launches.max():,} launches")
print(f"First launch: {df['LaunchYear'].min()}")
print(f"Latest launch: {df['LaunchYear'].max()}")
print(f"\nLaunches by Object Type:")
print(df['ObjectType'].value_counts())

## 4.6 Categorical Variable Analysis

*Examining frequency distributions and patterns in categorical variables*

In [0]:
# Object Type Distribution
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Count plot
ax1 = axes[0]
object_counts = df['ObjectType'].value_counts()
sns.barplot(x=object_counts.values, y=object_counts.index, ax=ax1, palette='viridis')
ax1.set_title('Object Type Distribution', fontsize=14, fontweight='bold')
ax1.set_xlabel('Count')
ax1.set_ylabel('Object Type')
for i, v in enumerate(object_counts.values):
    ax1.text(v + 500, i, f'{v:,}', va='center')

# Pie chart
ax2 = axes[1]
ax2.pie(object_counts.values, labels=object_counts.index, autopct='%1.1f%%', 
        startangle=90, colors=sns.color_palette('viridis', len(object_counts)))
ax2.set_title('Object Type Proportion', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

print("\n=== Object Type Summary ===")
print(object_counts)
print(f"\nTotal: {object_counts.sum():,}")

In [0]:
# Operational Status Distribution
plt.figure(figsize=(12, 6))
status_counts = df['OperationalStatus'].value_counts()
sns.barplot(x=status_counts.index, y=status_counts.values, palette='coolwarm')
plt.title('Operational Status Distribution', fontsize=14, fontweight='bold')
plt.xlabel('Operational Status')
plt.ylabel('Count')
plt.xticks(rotation=45)

# Add value labels
for i, v in enumerate(status_counts.values):
    plt.text(i, v + 500, f'{v:,}', ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

print("\n=== Operational Status Summary ===")
print(status_counts)
print(f"\nActive Objects: {status_counts.get('+', 0):,}")
print(f"Decayed Objects: {status_counts.get('D', 0):,}")

In [0]:
# Top 15 Countries by Launch Volume
plt.figure(figsize=(14, 8))
top_owners = df['Owner'].value_counts().head(15)
sns.barplot(y=top_owners.index, x=top_owners.values, palette='rocket')
plt.title('Top 15 Countries/Organizations by Object Count', fontsize=14, fontweight='bold')
plt.xlabel('Number of Objects')
plt.ylabel('Owner')

# Add value labels
for i, v in enumerate(top_owners.values):
    plt.text(v + 100, i, f'{v:,}', va='center')

plt.tight_layout()
plt.show()

print("\n=== Top 15 Owners ===")
for i, (owner, count) in enumerate(top_owners.items(), 1):
    print(f"{i:2d}. {owner:35s}: {count:6,} objects ({count/len(df)*100:5.2f}%)")

In [0]:
# Top Launch Sites
plt.figure(figsize=(14, 7))
top_sites = df['LaunchSite'].value_counts().head(10)
sns.barplot(y=top_sites.index, x=top_sites.values, palette='mako')
plt.title('Top 10 Launch Sites by Object Count', fontsize=14, fontweight='bold')
plt.xlabel('Number of Launches')
plt.ylabel('Launch Site')

# Add value labels
for i, v in enumerate(top_sites.values):
    plt.text(v + 100, i, f'{v:,}', va='center')

plt.tight_layout()
plt.show()

print("\n=== Top 10 Launch Sites ===")
for i, (site, count) in enumerate(top_sites.items(), 1):
    print(f"{i:2d}. {site:40s}: {count:6,} launches")

In [0]:
# Orbit State Distribution
plt.figure(figsize=(12, 6))
orbit_state_counts = df['OrbitState'].value_counts()
sns.barplot(x=orbit_state_counts.index, y=orbit_state_counts.values, palette='Set2')
plt.title('Orbit State Distribution', fontsize=14, fontweight='bold')
plt.xlabel('Orbit State')
plt.ylabel('Count')
plt.xticks(rotation=45, ha='right')

# Add value labels
for i, v in enumerate(orbit_state_counts.values):
    plt.text(i, v + 500, f'{v:,}', ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

print("\n=== Orbit State Summary ===")
print(orbit_state_counts)

## 4.7 Domain-Specific Insights

*Space debris analysis: orbital distribution, debris concentration, and historical trends*

In [0]:
# Orbital Class Distribution (using the binned OrbitClass column)
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Bar chart
ax1 = axes[0]
orbit_class_counts = df['OrbitClass'].value_counts()
sns.barplot(x=orbit_class_counts.index, y=orbit_class_counts.values, ax=ax1, palette='plasma')
ax1.set_title('Objects by Orbital Class', fontsize=14, fontweight='bold')
ax1.set_xlabel('Orbital Class')
ax1.set_ylabel('Count')
ax1.tick_params(axis='x', rotation=45)

# Add value labels
for i, v in enumerate(orbit_class_counts.values):
    ax1.text(i, v + 500, f'{v:,}', ha='center', fontweight='bold')

# Pie chart
ax2 = axes[1]
ax2.pie(orbit_class_counts.values, labels=orbit_class_counts.index, autopct='%1.1f%%',
        startangle=90, colors=sns.color_palette('plasma', len(orbit_class_counts)))
ax2.set_title('Orbital Class Distribution', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

print("\n=== Orbital Class Summary ===")
for orbit, count in orbit_class_counts.items():
    print(f"{orbit:25s}: {count:6,} objects ({count/len(df)*100:5.2f}%)")

In [0]:
# Debris vs Active Payloads Over Time
plt.figure(figsize=(14, 7))

# Create a simplified category
df['SimplifiedType'] = df['ObjectType'].replace({
    'Payload': 'Payload',
    'Debris': 'Debris',
    'Rocket Body': 'Rocket Body',
    'Unknown': 'Unknown'
})

# Group by year and simplified type
trend_data = df.groupby(['LaunchYear', 'SimplifiedType']).size().unstack(fill_value=0)

# Plot
trend_data.plot(kind='area', stacked=True, alpha=0.7, figsize=(14, 7), 
                colormap='tab10', linewidth=2)

plt.title('Debris vs Payloads vs Rocket Bodies Over Time', fontsize=14, fontweight='bold')
plt.xlabel('Launch Year', fontsize=12)
plt.ylabel('Object Count', fontsize=12)
plt.legend(title='Object Category', loc='upper left')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Summary statistics
print("\n=== Object Category Summary ===")
print(df['SimplifiedType'].value_counts())
print(f"\nDebris Percentage: {df[df['SimplifiedType']=='Debris'].shape[0]/len(df)*100:.2f}%")
print(f"Active Payloads: {df[(df['SimplifiedType']=='Payload') & (df['IsActive']==1)].shape[0]:,}")
print(f"Decayed Payloads: {df[(df['SimplifiedType']=='Payload') & (df['IsDecayed']==1)].shape[0]:,}")

In [0]:
# Top 10 Countries by Debris Count
plt.figure(figsize=(14, 7))

debris_by_owner = df[df['ObjectType'] == 'Debris'].groupby('Owner').size().sort_values(ascending=False).head(10)

sns.barplot(y=debris_by_owner.index, x=debris_by_owner.values, palette='Reds_r')
plt.title('Top 10 Countries/Organizations by Debris Count', fontsize=14, fontweight='bold')
plt.xlabel('Number of Debris Objects')
plt.ylabel('Owner')

# Add value labels
for i, v in enumerate(debris_by_owner.values):
    plt.text(v + 50, i, f'{v:,}', va='center')

plt.tight_layout()
plt.show()

print("\n=== Top 10 Debris Producers ===")
total_debris = df[df['ObjectType'] == 'Debris'].shape[0]
for i, (owner, count) in enumerate(debris_by_owner.items(), 1):
    pct = count / total_debris * 100
    print(f"{i:2d}. {owner:35s}: {count:5,} debris objects ({pct:5.2f}% of total debris)")

In [0]:
# Active vs Decayed by Object Type
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Create status labels
df['Status'] = df['IsDecayed'].map({1: 'Decayed', 0: 'Active'})

# Stacked bar chart
ax1 = axes[0]
status_by_type = pd.crosstab(df['ObjectType'], df['Status'])
status_by_type.plot(kind='bar', stacked=True, ax=ax1, color=['#2ecc71', '#e74c3c'])
ax1.set_title('Active vs Decayed Objects by Type', fontsize=14, fontweight='bold')
ax1.set_xlabel('Object Type')
ax1.set_ylabel('Count')
ax1.legend(title='Status')
ax1.tick_params(axis='x', rotation=45)

# Percentage bar chart
ax2 = axes[1]
status_pct = status_by_type.div(status_by_type.sum(axis=1), axis=0) * 100
status_pct.plot(kind='bar', stacked=True, ax=ax2, color=['#2ecc71', '#e74c3c'])
ax2.set_title('Active vs Decayed Objects (Percentage)', fontsize=14, fontweight='bold')
ax2.set_xlabel('Object Type')
ax2.set_ylabel('Percentage')
ax2.legend(title='Status')
ax2.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

print("\n=== Active vs Decayed Summary ===")
print(status_by_type)
print(f"\nTotal Active: {df[df['IsDecayed']==0].shape[0]:,}")
print(f"Total Decayed: {df[df['IsDecayed']==1].shape[0]:,}")

## 4.8 Multivariate Analysis

*Examining relationships across multiple variables simultaneously*

In [0]:
# Pair plot for key numerical variables
import warnings
warnings.filterwarnings('ignore')

# Select key variables for pair plot
key_vars = ['OrbitalPeriodMin', 'InclinationDegrees', 'MaxAltitudeKM', 'MinAltitudeKM', 'RadarSizeSQM']

# Sample data for performance (pair plots are expensive)
df_sample = df[key_vars + ['ObjectType']].sample(n=min(5000, len(df)), random_state=42)

# Create pair plot
sns.pairplot(df_sample, hue='ObjectType', palette='Set1', 
             diag_kind='kde', plot_kws={'alpha': 0.6, 's': 20},
             height=2.5, aspect=1)

plt.suptitle('Pair Plot: Relationships Between Orbital Parameters', y=1.02, fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

print("\n=== Pair Plot Summary ===")
print(f"Sample size: {len(df_sample):,} objects (from {len(df):,} total)")
print(f"Variables analyzed: {', '.join(key_vars)}")

# Phase 5: Feature Engineering & Preprocessing

## 4. Data Normalization / Feature Scaling

#### Note: Feature Scaling Not Applied 
- Feature scaling is not applied to this project as we not going to do any Machine learning part.

### When to Use Feature Scaling (For ML Modeling)

*Rescales continuous numerical features with large variations (e.g., altitudes reaching 36,000km vs. orbital periods of 90minutes) so algorithms treat them equally.*

* **`OrbitalPeriodMinutes`**
* **`InclinationDegrees`**
* **`ApogeeKM`**
* **`PerigeeKM`**
* **`RadarCrossSectionSQM`**



#### Note: Scaling Methods for ML

* **`RobustScaler`**
* **When to use:** When data has **heavy/extreme outliers**.
* **Formula:**
$$x_{\text{scaled}} = \frac{x - \text{Median}}{\text{IQR}}$$


*Where IQR = Q3 (75th percentile) - Q1 (25th percentile)*


* **`MinMaxScaler`**
* **When to use:** When we need a **values in [0, 1] range** and have **no outliers**.
* **Formula:**
$$x_{\text{scaled}} = \frac{x - x_{\text{min}}}{x_{\text{max}} - x_{\text{min}}}$$




* **`StandardScaler`**
* **When to use:** When data follows a **bell curve (normal distribution)** with **few or no outliers**.
* **Formula:**
$$x_{\text{scaled}} = \frac{x - \mu}{\sigma}$$


mean is $${\mu}$$ 
standard deviation is $$\sigma$$



---

### Code

```python
from sklearn.preprocessing import MinMaxScaler, RobustScaler, StandardScaler

```

## 5. Data Binning (Categorical Grouping)

#### Note:

**When to Bin in Python**

* If you're building **machine learning models** → bin in Python
* If you're making **charts in Python** (matplotlib, seaborn, Plotly) → bin in Python
* If you have **fixed business rules** that never change → bin in Python

---

**When NOT to Bin (Let Power BI Do It)**

* If your **final dashboard is in Power BI** → don't bin in Python
* If you want **smaller file sizes** → don't bin in Python
* If users need to **change bin ranges** in the report → don't bin in Python
* If you need **exact numbers later** → don't bin in Python

#### Column: MaxAltitudeKM
* **-1 - 2000:** Low Earth Orbit (LEO)
* **2000 - 35785:** Medium Earth Orbit (MEO)
* **35785 - 36000:** Geostationary Earth Orbit (GEO)
* **36000 - np.inf:** High Earth Orbit (HEO)

In [0]:

# 1. Orbital Altitude Bins (ApogeeKM)
# low -1 to 2000, medium 2000 to 35785, geo 35785 to 36000, Heo 36000 to infinity (float('inf))
orbit_bins = [-1, 2000, 35785, 36000, np.inf]
orbit_labels = ['Low Earth Orbit', 'Medium Earth Orbit', 'Geostationary Orbit', 'High Earth Orbit']
df['OrbitClass'] = pd.cut(df['MaxAltitudeKM'], bins=orbit_bins, labels=orbit_labels)

df

#### Column: RadarCrossSectionSQM
* **-1 - 0.1**: Small
* **0.1 - 1.0**: Medium
* **1.0 - np.inf**: Large

In [0]:
# Column: RadarCrossSectionSQM

size_bins = [-1, 0.1, 1.0, np.inf]
size_labels = ['Small', 'Medium', 'Large']
df['RadarSize'] = pd.cut(df['RadarSizeSQM'], bins=size_bins, labels=size_labels)
df.head(4)

#### Column: LaunchDate
* **1980-1900:** *19th Century*
* **1900–2000:** *20th century*
* **2000–Present:** *21st century*

In [0]:
# Column: LaunchDate 

launch_bins = [1800, 1900, 2000, 2035]
launch_labels = ['19th Century', '20th Century', '21st Century']
df['LaunchEra'] = pd.cut(df['LaunchDate'].dt.year, bins=launch_bins, labels=launch_labels)

## 6. Creating Indicator Variables

### Binary Flags for Power BI

*Creates binary (1 or 0) flags to easily isolate specific conditions in Power BI measures or ML features.*

* **`IsDecayed`:** `1` if `DecayDate` is present (or `OrbitState == "IMP"`), `0` if still in orbit.
* **`IsActive`:** `1` if `OperationalStatus == "+"` (or `ObjectType == "PAY"` and `DecayDate` is null), `0` otherwise.
* **`IsDebris`:** `1` if `ObjectType == "DEB"`, `0` otherwise.
* **`IsRocketBody`:** `1` if `ObjectType == "R/B"`, `0` otherwise.



In [0]:
import numpy as np

# 1. IsDecayed: 1 if DecayDate is present or OrbitState is 'IMP', 0 if still in orbit
df["IsDecayed"] = np.where(df["DecayDate"].notna() | (df["OrbitState"] == "IMP"), 1, 0)

# 2. IsActive: 1 if OperationalStatus is '+' or (ObjectType is 'PAY' and DecayDate is NaT), 0 otherwise
df["IsActive"] = np.where((df["OperationalStatus"] == "+") | ((df["ObjectType"] == "PAY") & (df["DecayDate"].isna())), 1, 0)

# 3. IsDebris: 1 if ObjectType is 'DEB', 0 otherwise
df["IsDebris"] = np.where(df["ObjectType"] == "DEB", 1, 0)

# 4. IsRocketBody: 1 if ObjectType is 'R/B', 0 otherwise
df["IsRocketBody"] = np.where(df["ObjectType"] == "R/B", 1, 0)

# Display the newly created binary columns to verify
display(df[["ObjectName", "ObjectType", "IsDecayed", "IsActive", "IsDebris", "IsRocketBody"]].head(10))

In [0]:
df.to_csv("/Workspace/Users/guruvendra47@gmail.com/Space-Debris-Project/raw/space_debris_cleaned.csv", index=False)
